In [1]:
from transformers import (
    AutoModelForSequenceClassification, 
    AutoTokenizer,
    Trainer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from torch.utils.data import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score

print("="*80)
print("COMPLETE FINE-TUNING EXAMPLE WITH DISTILGPT-2")
print("="*80)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-05 20:27:42.175387: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-05 20:27:48.149991: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-05 20:27:54.777039: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see sli

COMPLETE FINE-TUNING EXAMPLE WITH DISTILGPT-2


In [2]:
# ============================================================================
# STEP 1: Load DistilGPT-2 (Easiest Model)
# ============================================================================

print("\n[1/5] Loading DistilGPT-2...")

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model.config.pad_token_id = tokenizer.pad_token_id

params = sum(p.numel() for p in model.parameters())
print(f"✓ DistilGPT-2 loaded")
print(f"  Parameters: {params:,} (82M)")
print(f"  Size: ~330 MB")
print(f"  Why easy: Small, fast, uses less memory")


[1/5] Loading DistilGPT-2...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at distilgpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ DistilGPT-2 loaded
  Parameters: 81,914,112 (82M)
  Size: ~330 MB
  Why easy: Small, fast, uses less memory


In [3]:
# ============================================================================
# STEP 2: Apply LoRA (Optional but Recommended)
# ============================================================================

print("\n[2/5] Applying LoRA for efficient training...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["c_attn"],  # DistilGPT-2 uses same architecture as GPT-2
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✓ LoRA applied")
print(f"  Trainable params: {trainable:,} ({100*trainable/params:.2f}%)")


[2/5] Applying LoRA for efficient training...


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


trainable params: 148,992 || all params: 82,063,104 || trainable%: 0.1816

✓ LoRA applied
  Trainable params: 148,992 (0.18%)


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
# ============================================================================
# STEP 3: Load IMDB Dataset (Small Subset for Speed)
# ============================================================================

print("\n[3/5] Loading IMDB dataset...")

dataset = load_dataset("imdb")

# Use small subset for fast training (increase for better results)
train_data = dataset['train'].shuffle(seed=42).select(range(1000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

print(f"✓ Dataset loaded (small subset for speed)")
print(f"  Train: {len(train_data)} samples")
print(f"  Test: {len(test_data)} samples")


[3/5] Loading IMDB dataset...
✓ Dataset loaded (small subset for speed)
  Train: 1000 samples
  Test: 500 samples


In [5]:
# ============================================================================
# STEP 4: Prepare Dataset
# ============================================================================

print("\n[4/5] Preparing dataset...")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(
    train_data['text'],
    train_data['label'],
    tokenizer,
    max_length=128  # Shorter = faster training
)

test_dataset = IMDBDataset(
    test_data['text'],
    test_data['label'],
    tokenizer,
    max_length=128
)

print("✓ Dataset prepared")


[4/5] Preparing dataset...
✓ Dataset prepared


In [ ]:
# ============================================================================
# STEP 5: Train!
# ============================================================================

print("\n[5/5] Training DistilGPT-2...")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {'accuracy': accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir='./results_distilgpt2',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    logging_steps=25,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting training...")
print("(Should take ~5-10 minutes on CPU, ~2-3 minutes on GPU)")
trainer.train()

results = trainer.evaluate()

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"\n✓ You just fine-tuned your first LLM!")


[5/5] Training DistilGPT-2...

Starting training...
(Should take ~5-10 minutes on CPU, ~2-3 minutes on GPU)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/anaconda/envs/azureml_py38/

Epoch,Training Loss,Validation Loss,Accuracy
1,0.687300,0.688059,0.554000
2,0.693900,0.687934,0.546000


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
